In [ ]:
import requests
import pandas as pd
import json
import time

In [ ]:
BANK_NAME = "SC"
BANK_CODE = "SC"

SC_URL = "https://www.standardchartered.co.kr/np/kr/prdctList"
REFERER_URL = "https://www.standardchartered.co.kr/np/kr/pl/et/InterestRateDeposit4P.jsp?utm_source=chatgpt.com"

COMMON_HEADERS = {
    "accept": "application/json, text/javascript, */*; q=0.01",
    "content-type": "application/json;charset=UTF-8",
    "origin": "https://www.standardchartered.co.kr",
    "referer": REFERER_URL,
    "user-agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    ),
    "x-requested-with": "XMLHttpRequest",
}

In [ ]:
def generate_month_end_dates(start_date="2004-01-31", end_date="2019-12-31"):
    dates = []
    current = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end = pd.to_datetime(end_date)

    while current <= end:
        dates.append(current.strftime("%Y%m%d"))
        current = current + pd.offsets.MonthEnd(1)

    return dates

In [ ]:
def build_sc_payload(target_yyyymmdd):
    payload = {
        "serviceID": "HP_AP_RM_FxRate.selectSearchRateList",
        "SCFB_MESSAGE_ID": "HP_DpH718I01_H718_IN",
        "IMS_TRAN_CODE": "TI1IBF01",
        "IN_CLASS_CODE": "H718",
        "JOB_TYPE": "GZ",
        "SERVICE_CODE": "718",
        "TSPassword": "111111",
        "UserID": "FIRST900",
        "referDate": target_yyyymmdd
    }
    return payload

In [ ]:
def fetch_sc_json(session, target_yyyymmdd, debug=False):
    payload = build_sc_payload(target_yyyymmdd)

    resp = session.post(
        SC_URL,
        headers=COMMON_HEADERS,
        json=payload,
        timeout=60
    )
    resp.raise_for_status()

    if debug:
        print("status:", resp.status_code)
        print("content-type:", resp.headers.get("content-type"))
        print(resp.text[:1000])

    return resp.json()

In [ ]:
def normalize_sc_rows(raw_json, target_yyyymmdd):
    columns = [
        "bank", "bank_code", "target_date",
        "currency", "maturity", "rate", "product"
    ]

    try:
        vector = raw_json["HP_DpH718I01_H718_OUT"]["ARR"]["vector"]
    except Exception:
        return pd.DataFrame(columns=columns)

    target_date_fmt = pd.to_datetime(
        target_yyyymmdd, format="%Y%m%d"
    ).strftime("%Y-%m-%d")

    maturity_map = {
        "normalAmt": "보통예금",
        "noticeAmt": "통지예금",
        "onevWeak": "1주",
        "oneMonth": "1개월",
        "twoMonth": "2개월",
        "threeMonth": "3개월",
        "fourMonth": "4개월",
        "fiveMonth": "5개월",
        "sixMonth": "6개월",
    }

    rows = []

    for item in vector:
        inner = item.get("HP_DpH718I01_H718_OUT_ARR", {})
        currency = inner.get("currName")
        adobe_gb = inner.get("adobeGb")

        # 사용자 친화적으로 표시
        if adobe_gb == "1":
            resident_label = "거주자"
        elif adobe_gb == "2":
            resident_label = "비거주자"
        else:
            resident_label = f"구분{adobe_gb}"

        for raw_key, maturity_label in maturity_map.items():
            raw_rate = inner.get(raw_key)

            rows.append({
                "bank": BANK_NAME,
                "bank_code": BANK_CODE,
                "target_date": target_date_fmt,
                "currency": currency,
                "maturity": maturity_label,
                "rate": pd.to_numeric(raw_rate, errors="coerce") / 1000,
                "product": f"외화예금 ({resident_label})"
            })

    df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
    return df

In [ ]:
session = requests.Session()
test_date = "20260330"

raw_json = fetch_sc_json(session, test_date, debug=True)
df_test = normalize_sc_rows(raw_json, test_date)

print("rows:", len(df_test))
print(df_test.head(20))

In [ ]:
print(type(raw_json))
print(raw_json.keys())

out = raw_json.get("HP_DpH718I01_H718_OUT", {})
print(out.keys())

arr = out.get("ARR", {})
print(arr.keys())

vector = arr.get("vector", [])
print("vector length:", len(vector))

if len(vector) > 0:
    print(vector[0].keys())
    print(vector[0]["HP_DpH718I01_H718_OUT_ARR"])

In [ ]:
def crawl_sc_range(start_date="2004-01-31", end_date="2019-12-31", sleep_sec=0.3):
    session = requests.Session()
    all_frames = []
    failed_dates = []

    date_list = generate_month_end_dates(start_date, end_date)
    print(f"총 {len(date_list)}개 날짜 수집 시작")

    for i, yyyymmdd in enumerate(date_list, 1):
        try:
            raw_json = fetch_sc_json(session, yyyymmdd, debug=False)
            df_day = normalize_sc_rows(raw_json, yyyymmdd)

            if not df_day.empty:
                all_frames.append(df_day)

            print(f"[{i}/{len(date_list)}] {yyyymmdd} 완료 - {len(df_day)} rows")

        except Exception as e:
            print(f"[{i}/{len(date_list)}] {yyyymmdd} 실패 - {e}")
            failed_dates.append({
                "target_date": yyyymmdd,
                "error": str(e)
            })

        time.sleep(sleep_sec)

    if all_frames:
        final_df = pd.concat(all_frames, ignore_index=True)
        final_df = final_df.drop_duplicates().reset_index(drop=True)
    else:
        final_df = pd.DataFrame(columns=[
            "bank", "bank_code", "target_date",
            "currency", "maturity", "rate", "product"
        ])

    failed_df = pd.DataFrame(failed_dates)
    return final_df, failed_df

In [ ]:
sc_df, failed_df = crawl_sc_range(
    start_date="2004-01-31",
    end_date="2019-12-31",
    sleep_sec=0.3
)

print("\n최종 shape:", sc_df.shape)
print(sc_df.head())
print("\n실패 건수:", len(failed_df))
print(failed_df.head())

In [ ]:
print("고유 날짜 수:", sc_df["target_date"].nunique())

print("\n날짜별 행 수:")
print(sc_df["target_date"].value_counts().head())

print("\n통화별 행 수:")
print(sc_df["currency"].value_counts().head(20))

print("\n상품별 행 수:")
print(sc_df["product"].value_counts())

print("\n만기별 행 수:")
print(sc_df["maturity"].value_counts())

print("\n샘플:")
print(sc_df.sample(min(20, len(sc_df)), random_state=42))

In [ ]:
output_file = "sc_2004_2019_full.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    sc_df.to_excel(writer, sheet_name="Sheet1", index=False)
    failed_df.to_excel(writer, sheet_name="failed_dates", index=False)

print(f"저장 완료: {output_file}")